In [1]:
import os
from utils import print_completion_stats, load_experiment_results

os.environ['OPENESTIMATE_ROOT'] = '/home/ubuntu/alana/openestimate/'

experiment_name = 'model_family_comparison'
datasets = ['glassdoor']
results_sets = [load_experiment_results(dataset, experiment_name) for dataset in datasets] 
for results, variables in results_sets: 
        print_completion_stats(results)

results = results_sets[0][0]
variables = results_sets[0][1]

Number of trials found:  1
hi
hii
Completion Statistics

Completion-rate by model (%):

Approach                                   Completion %
-----------------------------------------------------------------
gpt-4o_base_direct_temp0.5                  100.0
meta-llama-3-70b_base_direct_temp0.5        100.0
meta-llama-3-8b_base_direct_temp0.5         100.0
o3-mini_base_direct_tempmedium              100.0
o4-mini_base_direct_tempmedium              100.0
qwen3-235b-a22b-fp8-tput_base_direct_temp0.6    100.0


# Load baselines 

In [2]:
import json 

baselines_file_path = os.path.expanduser("{}data/baselines/{}_baselines_output.json".format(os.environ['OPENESTIMATE_ROOT'], 'glassdoor')) 
stat_baselines = json.load(open(baselines_file_path, 'r'))


In [3]:
import pandas as pd 
from scipy.stats import beta as beta_dist, norm as normal_dist


stat_baselines_metrics_by_var = {}
for var, info in stat_baselines.items():
    stat_baselines_metrics_by_var[var] = {}
    for n, baseline_infos in info.items():
        var_name = variables[var]['variable']

        # Store each resampling's metrics separately
        resampling_results = []

        # Determine if this is a lognormal baseline
        is_lognormal = 'lognorm' in str(n)
        # Extract the numeric sample size (e.g., "5" from "5_lognorm" or "5" from "5")
        n_numeric = str(n).replace('_lognorm', '')

        # Process each resampling
        trial = 0
        for baseline_info in baseline_infos:
            var_type = 'beta' if 'alpha' in baseline_info else 'gaussian'
            var_difficulty = 'easy' if 'easy' in var else 'medium' if 'medium' in var else 'hard' if 'hard' in var else 'base'
            ground_truth = variables[var]['mean']

            # Determine ground truth distribution type based on the baseline type
            if is_lognormal:
                ground_truth_dist = 'lognormal'
            elif var_type == 'gaussian':
                ground_truth_dist = 'gaussian'
            else:
                ground_truth_dist = 'beta'

            # Store each resampling's results separately
            resampling_results.append({
                'variable_name': var_name,
                'variable': var,
                'ground_truth': ground_truth,
                'ground_truth_distribution_type': ground_truth_dist,
                'fitted_distribution_type': ground_truth_dist,
                'mu': baseline_info.get('mu') if var_type == 'gaussian' else None,
                'sigma': baseline_info.get('sigma', baseline_info.get('std')) if var_type == 'gaussian' else None,
                'a': baseline_info['alpha'] if 'alpha' in baseline_info else None,
                'b': baseline_info['beta'] if 'beta' in baseline_info else None,
                'trial': trial,
                'difficulty': var_difficulty,
                'sample_size': n_numeric
            })
            trial += 1
        # Store all resampling results for this sample size
        stat_baselines_metrics_by_var[var][n] = resampling_results

flattened_metrics = []
for var, baselines in stat_baselines_metrics_by_var.items():
    for n, resamplings in baselines.items():
        # Extract numeric sample size (works for both "5" and "5_lognorm")
        n_numeric = str(n).replace('_lognorm', '')

        for resampling in resamplings:
            resampling_copy = resampling.copy()
            resampling_copy['approach'] = f'statistical_baseline_n{n_numeric}'
            flattened_metrics.append(resampling_copy)

results_df = pd.DataFrame(flattened_metrics)

results = pd.concat([results, results_df], ignore_index=True)

In [4]:
results_df['ground_truth_distribution_type'].value_counts()

ground_truth_distribution_type
gaussian     4646
lognormal    4646
Name: count, dtype: int64

In [5]:
import pandas as pd 

# mask = results['fitted_distribution_type'] == 'gaussian'
# results.loc[mask, 'mu'] = results.loc[mask, 'mean']
# results.loc[mask, 'sigma'] = results.loc[mask, 'std']


# results[(results['fitted_distribution_type'] == 'gaussian')].head()
# results = results.drop('mean', axis=1)
# results = results.drop('std', axis=1)

# Update ground truth column to be chosen based on fitted distribution for LLMs 

In [6]:
mask = results['fitted_distribution_type'] == 'lognormal'

results.loc[mask, 'ground_truth'] = results.loc[mask].apply(
    lambda row: variables[row['variable']].get('mean_lognormal', variables[row['variable']]['mean']), 
    axis=1
)

# Add mean, median, mode of each distribution 

In [7]:
import numpy as np
from scipy import stats


def compute_normal_mean_median_mode(mu, sigma):
    return {'mean': mu, 'median': mu, 'mode': mu}


def compute_beta_mean_median_mode(a, b): 
    # Mean of Beta(a, b) = a / (a + b)
    mean = a / (a + b)
    
    # Median - no closed form, use scipy
    median = stats.beta.median(a, b)
    
    if a > 1 and b > 1:
        mode = (a - 1) / (a + b - 2)
    elif a == 1 and b == 1:
        mode = np.nan  # uniform distribution
    elif a <= 1 and b > 1:
        mode = 0.0
    elif a > 1 and b <= 1:
        mode = 1.0
    else:  # a < 1 and b < 1 (bimodal)
        mode = np.nan  # or could return [0.0, 1.0

    return {
        'mean': mean,
        'median': median,
        'mode': mode  
    }


def compute_lognormal_mean_median_mode(mu, sigma):
    """
    Compute mean, median, and mode of a Lognormal distribution.
    
    Parameters:
    -----------
    mu : float
        Mean of the underlying normal distribution
    sigma : float
        Standard deviation of the underlying normal distribution, must be > 0
    
    Returns:
    --------
    dict with keys 'mean', 'median', 'mode'
    """
    import numpy as np
    
    # Mean of Lognormal(mu, sigma) = exp(mu + sigma^2/2)
    mean = np.exp(mu + sigma**2 / 2)
    
    # Median of Lognormal(mu, sigma) = exp(mu)
    median = np.exp(mu)
    
    # Mode of Lognormal(mu, sigma) = exp(mu - sigma^2)
    mode = np.exp(mu - sigma**2)
    
    return {
        'mean': mean,
        'median': median,
        'mode': mode
    }

In [8]:
beta_vars = results['fitted_distribution_type'] == 'beta'
lognorm_vars = results['fitted_distribution_type'] == 'lognormal'
norm_vars = results['fitted_distribution_type'] == 'gaussian'

results['mean'] = np.nan
results['median'] = np.nan 
results['mode'] = np.nan 

results.loc[beta_vars, ['mean', 'median', 'mode']] = results.loc[beta_vars].apply(
    lambda row: pd.Series(compute_beta_mean_median_mode(row['a'], row['b'])), axis=1)

results.loc[lognorm_vars, ['mean', 'median', 'mode']] = results.loc[lognorm_vars].apply(
    lambda row: pd.Series(compute_lognormal_mean_median_mode(row['mu'], row['sigma'])), axis=1)

results.loc[norm_vars, ['mean', 'median', 'mode']] = results.loc[norm_vars].apply(
    lambda row: pd.Series(compute_normal_mean_median_mode(row['mu'], row['sigma'])), axis=1)

# Compute AE using all central tendencies for each variable 

In [9]:
variables

{'Midpoint Salary': {'mean': 125685.70774091627,
  'std': 31627.035629085392,
  'se': 1257.061810741376,
  'mean_lognormal': 126461.03222464297,
  'std_lognormal': 36308.458685018115,
  'base_variable': 'Midpoint Salary',
  'ground_truth_distribution_type': 'normal',
  'variable': 'The average midpoint of the posted salary range (in dollars) for data science and adjacent jobs in the US',
  'conditions': [],
  'nat_langs': []},
 'single_0': {'mean': 133754.13194444444,
  'std': 32425.307908670307,
  'se': 1910.6795920235472,
  'mean_lognormal': 133694.84959063056,
  'std_lognormal': 31564.47866618305,
  'base_variable': 'Midpoint Salary',
  'conditions': [['IsPublic', 'public']],
  'nat_langs': ['the company is public'],
  'difficulty': 'single',
  'ground_truth_distribution_type': 'normal',
  'variable': 'The average midpoint of the posted salary range (in dollars) for data science and adjacent jobs in the US, given the company is public'},
 'single_1': {'mean': 115062.5,
  'std': 1670

In [10]:
results['abs_error_from_mean'] = np.abs(results['ground_truth'] - results['mean'])
results['abs_error_from_median'] = np.abs(results['ground_truth'] - results['median'])
results['abs_error_from_mode'] = np.abs(results['ground_truth'] - results['mode'])

# Determine quartile of ground truth

In [11]:
def get_quartiles_from_gaussian(mean, std):
    return normal_dist.ppf([0.25, 0.5, 0.75], mean, std)


def get_quartiles_from_beta(alpha, beta_param):
    res = beta_dist.ppf([0.25, 0.5, 0.75], alpha, beta_param)
    return res


def get_quartiles_from_lognormal(mu, sigma):
    return np.exp(stats.norm.ppf([0.25, 0.5, 0.75], mu, sigma))


def determine_quartile_of_gt(results):
    # Initialize the 'quartile_of_gt' column with default values
    results['quartile_of_gt'] = np.nan 

    for index, row in results.iterrows():
        if row['fitted_distribution_type'] == 'normal' or row['fitted_distribution_type'] == 'gaussian':
            if row['mean'] is not None:
                quartiles = get_quartiles_from_gaussian(row['mu'], row['sigma'])
        elif row['fitted_distribution_type'] == 'beta' or row['fitted_distribution_type'] == 'binomial':
            # print("Alpha: ", row['alpha'], "Beta: ", row['beta'])
            try: 
                alpha = row['alpha']
                beta = row['beta']
            except:
                alpha = row['a']
                beta = row['b']

            if alpha == 0: 
                alpha = 0.00001
            if beta == 0:
                beta = 0.00001
            if alpha == 0:
                alpha = 0.00001
            quartiles = get_quartiles_from_beta(alpha, beta)
        elif row['fitted_distribution_type'] == 'lognormal':
            if 'sigma' in row:
                quartiles = get_quartiles_from_lognormal(row['mu'], row['sigma'])
            else: 
                print(row)
                quartiles = get_quartiles_from_lognormal(row['mu'], row['std']) 
        else:
            if row['ground_truth_distribution_type'] == 'gaussian':
                quartiles = get_quartiles_from_gaussian(row['mu'], row['sigma'])
            elif row['ground_truth_distribution_type'] == 'beta':
                try: 
                    alpha = row['alpha']
                    beta = row['beta']
                except:
                    alpha = row['a']
                    beta = row['b']

                if alpha == 0: 
                    alpha = 0.00001
                if beta == 0:
                    beta = 0.00001
                quartiles = get_quartiles_from_beta(alpha, beta)
            else:
                raise ValueError("Unknown distribution type: ", row['ground_truth_distribution_type'])
            
        if row['ground_truth'] < quartiles[0]: 
            quartile_of_gt = 1
        elif row['ground_truth'] > quartiles[0] and row['ground_truth'] < quartiles[1]:
            quartile_of_gt = 2
        elif row['ground_truth'] > quartiles[1] and row['ground_truth'] < quartiles[2]:
            quartile_of_gt = 3
        elif row['ground_truth'] > quartiles[2]:
            quartile_of_gt = 4
        else: 
            quartile_of_gt = None
        results.at[index, 'quartile_of_gt'] = quartile_of_gt
    return results


results = determine_quartile_of_gt(results)

# Compute stdevs 

In [12]:
# Compute std for each distribution type
def compute_normal_std(mu, sigma):
    return sigma

def compute_beta_std(a, b):
    return stats.beta.std(a, b)

def compute_lognormal_std(mu, sigma):
    """
    Compute std of a Lognormal distribution.
    
    For Lognormal(mu, sigma):
    Var(X) = (exp(sigma^2) - 1) * exp(2*mu + sigma^2)
    Std(X) = sqrt(Var(X))
    """
    variance = (np.exp(sigma**2) - 1) * np.exp(2*mu + sigma**2)
    return np.sqrt(variance)

# Compute std for all rows
results['std'] = np.nan

results.loc[beta_vars, 'std'] = results.loc[beta_vars].apply(
    lambda row: compute_beta_std(row['a'], row['b']), axis=1)

results.loc[lognorm_vars, 'std'] = results.loc[lognorm_vars].apply(
    lambda row: compute_lognormal_std(row['mu'], row['sigma']), axis=1)

results.loc[norm_vars, 'std'] = results.loc[norm_vars].apply(
    lambda row: compute_normal_std(row['mu'], row['sigma']), axis=1)

# Compute error ratios

In [13]:
fallback_count = 0
exact_match_count = 0

results['error_ratio_mean'] = np.nan
results['error_ratio_median'] = np.nan
results['error_ratio_mode'] = np.nan
results['std_ratio'] = np.nan
results['associated_baseline_error_mean'] = np.nan
results['associated_baseline_error_median'] = np.nan
results['associated_baseline_error_mode'] = np.nan
results['associated_baseline_std'] = np.nan

for idx, row in results.iterrows():
    if "statistical" not in row["approach"]:
        # First try: Match baselines with same variable, sample_size=5, and distribution type
        baselines = results[
            (results["approach"].str.contains("statistical", na=False)) & 
            (results["sample_size"] == "5") &
            (results["variable"] == row["variable"]) & 
            (results["ground_truth_distribution_type"] == row["fitted_distribution_type"])
        ]
        
        # Fallback: If no exact match, use any available baseline for this variable
        if len(baselines) == 0:
            baselines = results[
                (results["approach"].str.contains("statistical", na=False)) & 
                (results["sample_size"] == "5") &
                (results["variable"] == row["variable"])
            ]
            if len(baselines) > 0:
                fallback_count += 1
        else:
            exact_match_count += 1
        
        if len(baselines) == 0:
            continue
            
        baseline_avg_error_mean = baselines["abs_error_from_mean"].mean()
        baseline_avg_error_median = baselines["abs_error_from_median"].mean()
        baseline_avg_error_mode = baselines["abs_error_from_mode"].mean()
        baseline_avg_std = baselines["std"].mean()

        error_ratio_mean = row["abs_error_from_mean"] / baseline_avg_error_mean 
        error_ratio_median = row["abs_error_from_median"] / baseline_avg_error_median
        error_ratio_mode = row["abs_error_from_mode"] / baseline_avg_error_mode
        std_ratio = row["std"] / baseline_avg_std

        results.at[idx, 'associated_baseline_error_mean'] = baseline_avg_error_mean
        results.at[idx, 'associated_baseline_error_median'] = baseline_avg_error_median
        results.at[idx, 'associated_baseline_error_mode'] = baseline_avg_error_mode
        results.at[idx, 'associated_baseline_std'] = baseline_avg_std
        results.at[idx, 'error_ratio_mean'] = error_ratio_mean
        results.at[idx, 'error_ratio_median'] = error_ratio_median
        results.at[idx, 'error_ratio_mode'] = error_ratio_mode
        results.at[idx, 'std_ratio'] = std_ratio
    
print(f"\nExact distribution type matches: {exact_match_count}")
print(f"Fallback to any available baseline: {fallback_count}")


Exact distribution type matches: 276
Fallback to any available baseline: 0


# Sanity check error ratio averages by model family

In [ ]:
# Export debug results to CSV
debug_columns = [
    'approach', 'variable', 'variable_name', 'fitted_distribution_type', 
    'ground_truth', 'mean', 'median', 'mode', 'mu', 'sigma','a', 'b', 'std', 
    'associated_baseline_std'
]

results[debug_columns].to_csv('debug_results.csv', index=False)

In [15]:
results.groupby('approach')[['error_ratio_mean', 'error_ratio_median', 'error_ratio_mode', 'std_ratio']].mean()

,error_ratio_mean,error_ratio_median,error_ratio_mode,std_ratio
approach,,,,
gpt-4o_base_direct_temp0.5,1.610540,1.554383,2.004962,5.479715
meta-llama-3-70b_base_direct_temp0.5,5.181329,5.065231,5.301792,4.169175
meta-llama-3-8b_base_direct_temp0.5,14.809358,14.502087,13.894514,0.565133
o3-mini_base_direct_tempmedium,1.704325,1.425466,1.667799,5.401377
o4-mini_base_direct_tempmedium,1.239848,1.097813,1.202205,3.621687
qwen3-235b-a22b-fp8-tput_base_direct_temp0.6,1.622940,1.501867,1.846298,3.930029
statistical_baseline_n10,NaN,NaN,NaN,NaN
statistical_baseline_n20,NaN,NaN,NaN,NaN
statistical_baseline_n30,NaN,NaN,NaN,NaN
